In [2]:
conda install -c conda-forge pillow pillow-heif imageio libheif

ValueError: The python kernel does not appear to be a conda environment.  Please use ``%pip install`` instead.

In [3]:
#!/usr/bin/env python3
# Auto image discovery + interactive HSV selection per image (HEIC-friendly)
# Batch-interleaved Green/Blue processing with safe post-processing of originals:
#   - Works in batches of BATCH_SIZE (default 15)
#   - For each batch: prompt -> GREEN -> save checkpoint -> prompt -> BLUE -> save checkpoint
#   - After an image has been processed for ALL colors, it is moved/trashed/deleted per DELETE_MODE
#
# UI/ergonomics:
#   - Large resizable window, fullscreen (F), reset (0)
#   - Mouse wheel zoom; right-button drag to pan
#   - Smaller wrapped overlay text
#   - **Option B**: fix window position to prevent cascading downward (cv2.moveWindow)
# Fatigue helpers:
#   - Press 'R' to reuse the last HSV range for the current color
#   - Press 'S' to skip current image

import os
import shutil
from pathlib import Path
from typing import Optional, Tuple, Dict, Any, List
import cv2
import numpy as np
import pandas as pd

# ========= CONFIG =========
#base_dir = r"I:\My Drive\2. Post-PhD\2. Research\13. Participant-driven survey\2. Data\2. ResizedTest"
#base_dir = r"C:\Users\chang\Downloads\Blue_Highlighted_PerImage-auto\blue missing"
base_dir = r"./output_cropped/"

# Resize target (w, h) for processing/preview (increase if you want more detail)
resize_dim = (300 * 3, 225 * 3)

# File types to include (case-insensitive)
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".heic", ".heif")
RECURSIVE = False  # set True if you want to traverse subfolders

# Batch size
BATCH_SIZE = 15

# ========= COLOR SETTINGS =========
color_configs = {
    "Green": {
        "initial_HSV": (35, 40, 40, 85, 255, 255),
        "highlight_color": [0, 255, 0],   # BGR
        "output_subdir": "Green_Highlighted_PerImage-auto"
    },
    "Blue": {
        "initial_HSV": (90, 60, 40, 140, 255, 255),
        "highlight_color": [255, 0, 0],   # BGR
        "output_subdir": "Blue_Highlighted_PerImage-auto"
    }
}
# Interleaving order per batch (change order if you prefer):
COLOR_ORDER: List[str] = ["Green", "Blue"]

# ========= POST-PROCESSING OF ORIGINALS =========
# What to do with an original file AFTER it has been processed for ALL colors:
#   "move"   -> move to Processed_Originals/ (preserve subfolders)  [SAFE DEFAULT]
#   "trash"  -> send to OS recycle bin (needs: pip install send2trash)
#   "delete" -> permanently delete (use with care)
#   "none"   -> do nothing
DELETE_MODE = "move"
PROCESSED_ROOT = "Processed_Originals"  # used only when DELETE_MODE == "move"

# ========= UI / WINDOW SETTINGS =========
WINDOW_W = 1400
WINDOW_H = 900
WINDOW_X = 50     # ← fixed position (from top-left of screen)
WINDOW_Y = 50
POPUP_X = 100     # popups fixed position
POPUP_Y = 80

FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5      # smaller text
TEXT_THICK = 1        # thinner stroke
LINE_GAP = 6          # gap between wrapped lines (pixels)
PADDING = 8           # padding inside text background box
PAN_STEP = 60         # pixels per arrow press (scaled by zoom)
TEXT_COLOR = (255, 255, 255)  # white
BG_COLOR = (0, 0, 0)          # black background for readability
BG_ALPHA = 0.35               # semi-transparent background

# ========= OPTIONAL HEIC SUPPORT (pillow-heif or imageio) =========
_PIL_OK = False
_HEIF_OK = False
_IIO_OK = False
try:
    from PIL import Image, ImageFile, ImageOps
    ImageFile.LOAD_TRUNCATED_IMAGES = True
    _PIL_OK = True
    try:
        import pillow_heif  # noqa: F401
        _HEIF_OK = True
    except Exception:
        _HEIF_OK = False
except Exception:
    _PIL_OK = False

try:
    import imageio.v3 as iio
    _IIO_OK = True
except Exception:
    _IIO_OK = False

# Optional recycle-bin support
_SEND2TRASH = False
try:
    from send2trash import send2trash
    _SEND2TRASH = True
except Exception:
    _SEND2TRASH = False


def read_image_any(path_str: str) -> Optional[np.ndarray]:
    """Read image into BGR uint8 (OpenCV-friendly), including HEIC when possible."""
    p = Path(path_str)
    ext = p.suffix.lower()

    if ext not in (".heic", ".heif"):
        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if img is None:
            if _PIL_OK:
                try:
                    pil = Image.open(str(p)).convert("RGB")
                    arr = np.array(pil)  # RGB
                    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
                except Exception:
                    return None
            return None
        if img.ndim == 3 and img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        return img

    # HEIC/HEIF:
    if _PIL_OK and _HEIF_OK:
        try:
            pil = Image.open(str(p))
            pil = ImageOps.exif_transpose(pil)
            pil = pil.convert("RGB")
            arr = np.array(pil)
            return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        except Exception:
            pass
    if _IIO_OK:
        try:
            arr = iio.imread(str(p))
            if arr.dtype != np.uint8:
                arr = arr.astype(np.uint8)
            if arr.ndim == 3 and arr.shape[2] == 4:
                arr = arr[:, :, :3]
            if arr.ndim == 2:
                arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
            else:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
            return arr
        except Exception:
            pass
    print(f"❌ HEIC reader not available for: {path_str} (install pillow-heif or imageio)")
    return None


def iter_images(root: str, recursive: bool):
    rootp = Path(root)
    files = (p for p in (rootp.rglob("*") if recursive else rootp.glob("*")) if p.is_file())
    for p in files:
        if p.suffix.lower() in EXTS:
            yield p


def nothing(x):
    pass


def save_checkpoint(results: list, output_dir: str, color_name: str, checkpoint_index: int):
    """Save checkpoint Excel + CSV after each batch for the given color."""
    if not results:
        return
    df = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    excel_path = os.path.join(output_dir, f"{color_name.lower()}_pixel_ratios_per_image_checkpoint_{checkpoint_index}.xlsx")
    csv_path = os.path.join(output_dir, f"{color_name.lower()}_pixel_ratios_per_image_checkpoint_{checkpoint_index}.csv")
    df.to_excel(excel_path, index=False)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"💾 Checkpoint {checkpoint_index} saved for {color_name}: {excel_path}")
    print(f"💾 Checkpoint {checkpoint_index} saved for {color_name}: {csv_path}")


def finalize_save(results: list, output_dir: str, color_name: str):
    """Final save Excel + CSV at the end for the given color."""
    if not results:
        return
    df = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    excel_path = os.path.join(output_dir, f"{color_name.lower()}_pixel_ratios_per_image.xlsx")
    csv_path = os.path.join(output_dir, f"{color_name.lower()}_pixel_ratios_per_image.csv")
    df.to_excel(excel_path, index=False)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"📊 Summary saved for {color_name} to: {excel_path}")
    print(f"📄 CSV saved for {color_name} to: {csv_path}")


def draw_wrapped_text(
    img: np.ndarray,
    text: str,
    topleft: Tuple[int, int],
    max_width: int,
    font: int,
    scale: float,
    color: Tuple[int, int, int],
    thickness: int,
    line_gap: int = 6,
    bg_color: Tuple[int, int, int] = (0, 0, 0),
    bg_alpha: float = 0.35,
    padding: int = 8
):
    """Draw word-wrapped text within max_width at topleft; adds semi-transparent background box."""
    x0, y0 = topleft
    words = text.split()
    lines = []
    cur = ""
    for w in words:
        test = (cur + " " + w).strip()
        (tw, th), _ = cv2.getTextSize(test, font, scale, thickness)
        if tw <= max_width or not cur:
            cur = test
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)

    line_sizes = [cv2.getTextSize(line, font, scale, thickness)[0] for line in lines]
    if not line_sizes:
        return
    block_w = min(max(w for w, _ in line_sizes), max_width)
    block_h = sum(h for _, h in line_sizes) + line_gap * (len(lines) - 1)

    x1 = x0 - padding
    y1 = y0 - padding
    x2 = x0 + block_w + padding
    y2 = y0 + block_h + padding

    overlay = img.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), bg_color, -1)
    cv2.addWeighted(overlay, bg_alpha, img, 1 - bg_alpha, 0, img)

    y = y0
    for line, (_, lh) in zip(lines, line_sizes):
        cv2.putText(img, line, (x0, y + lh), font, scale, color, thickness, cv2.LINE_AA)
        y += lh + line_gap


def show_popup(message: str, title: str = "Notice", w: int = 900, h: int = 220, wait_ms: int = 0):
    """Simple popup using OpenCV to notify user between batches. Positioned consistently."""
    canvas = np.full((h, w, 3), 30, dtype=np.uint8)
    cv2.putText(canvas, title, (20, 50), FONT, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    draw_wrapped_text(
        canvas, message, topleft=(20, 80), max_width=w - 40,
        font=FONT, scale=0.6, color=(255, 255, 255), thickness=1,
        line_gap=8, bg_color=(0, 0, 0), bg_alpha=0.0, padding=0
    )
    win = f"__popup__{title}"
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, w, h)
    cv2.moveWindow(win, POPUP_X, POPUP_Y)  # ← keep popups from cascading
    cv2.imshow(win, canvas)
    cv2.waitKey(wait_ms)  # 0 = wait for key
    cv2.destroyWindow(win)


def safe_make_unique(dest_path: Path) -> Path:
    """Ensure destination path is unique by appending (1), (2), ... if needed."""
    if not dest_path.exists():
        return dest_path
    stem = dest_path.stem
    suffix = dest_path.suffix
    parent = dest_path.parent
    i = 1
    while True:
        candidate = parent / f"{stem} ({i}){suffix}"
        if not candidate.exists():
            return candidate
        i += 1


def postprocess_file(original_path: Path):
    """
    Act on the original file after it has been processed for ALL colors.
    Honors DELETE_MODE:
      - move: move to base_dir/Processed_Originals/(relative path)
      - trash: send to recycle bin
      - delete: permanent delete
      - none: do nothing
    """
    mode = DELETE_MODE.lower()
    try:
        if mode == "none":
            return

        if mode == "move":
            rel = original_path.relative_to(base_dir) if str(original_path).startswith(str(base_dir)) else Path(original_path.name)
            target_root = Path(base_dir) / PROCESSED_ROOT
            target_path = target_root / rel
            target_path.parent.mkdir(parents=True, exist_ok=True)
            unique_dest = safe_make_unique(target_path)
            shutil.move(str(original_path), str(unique_dest))
            print(f"📁 Moved original to: {unique_dest}")

        elif mode == "trash":
            if not _SEND2TRASH:
                print("⚠️ send2trash not installed; falling back to move.")
                rel = original_path.relative_to(base_dir) if str(original_path).startswith(str(base_dir)) else Path(original_path.name)
                target_root = Path(base_dir) / PROCESSED_ROOT
                target_path = target_root / rel
                target_path.parent.mkdir(parents=True, exist_ok=True)
                unique_dest = safe_make_unique(target_path)
                shutil.move(str(original_path), str(unique_dest))
                print(f"📁 Moved original to: {unique_dest}")
            else:
                send2trash(str(original_path))
                print(f"🗑️ Sent to recycle bin: {original_path}")

        elif mode == "delete":
            original_path.unlink(missing_ok=True)
            print(f"🗑️ Permanently deleted: {original_path}")

        else:
            print(f"⚠️ Unknown DELETE_MODE '{DELETE_MODE}', skipping deletion.")
    except Exception as e:
        print(f"❌ Post-process failed for {original_path}: {e}")


def process_image_for_color(
    img_bgr: np.ndarray,
    img_path: str,
    color_name: str,
    config: dict,
    output_dir: str,
    last_range: Optional[Tuple[np.ndarray, np.ndarray]]
) -> Tuple[Optional[Dict[str, Any]], Optional[Tuple[np.ndarray, np.ndarray]]]:
    """Interactive picker with ergonomics; returns (row_dict or None, updated_last_range)."""
    name_stem = Path(img_path).stem
    ext = Path(img_path).suffix
    img = cv2.resize(img_bgr, resize_dim, interpolation=cv2.INTER_AREA)
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    clicked_hsvs = []

    # Per-image window (Option B): create + size + **pin position** to avoid cascading
    window_name = f"Click HSV: {name_stem} [{color_name}]"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, WINDOW_W, WINDOW_H)
    cv2.moveWindow(window_name, WINDOW_X, WINDOW_Y)  # ← fixed position

    cv2.createTrackbar("Hue ±", window_name, 10, 50, nothing)
    cv2.createTrackbar("Sat ±", window_name, 40, 127, nothing)
    cv2.createTrackbar("Val ±", window_name, 40, 127, nothing)

    zoom = 1.0
    offset_x, offset_y = 0, 0
    fullscreen = False

    dragging = False
    drag_start = (0, 0)
    start_offset = (0, 0)

    def clamp_offset(ox, oy, view_w, view_h, w, h):
        ox = max(0, min(w - view_w, ox))
        oy = max(0, min(h - view_h, oy))
        return ox, oy

    def mouse_callback(event, x, y, flags, param):
        nonlocal offset_x, offset_y, zoom, dragging, drag_start, start_offset

        h, w = img.shape[:2]
        view_w = int(w / zoom)
        view_h = int(h / zoom)
        x1 = max(0, min(w - view_w, offset_x))
        y1 = max(0, min(h - view_h, offset_y))

        if event == cv2.EVENT_LBUTTONDOWN:
            true_x = int(x / zoom + x1)
            true_y = int(y / zoom + y1)
            hsv_val = hsv_img[true_y, true_x]
            clicked_hsvs.append(hsv_val)
            print(f"🎯 Picked HSV #{len(clicked_hsvs)} @ ({true_x}, {true_y}): {hsv_val}")

        elif event == cv2.EVENT_RBUTTONDOWN:
            dragging = True
            drag_start = (x, y)
            start_offset = (offset_x, offset_y)

        elif event == cv2.EVENT_MOUSEMOVE and dragging:
            dx = x - drag_start[0]
            dy = y - drag_start[1]
            new_off_x = start_offset[0] - int(dx / zoom)
            new_off_y = start_offset[1] - int(dy / zoom)
            offset_x, offset_y = clamp_offset(new_off_x, new_off_y, view_w, view_h, w, h)

        elif event == cv2.EVENT_RBUTTONUP:
            dragging = False

        elif event == cv2.EVENT_MOUSEWHEEL:
            delta = 1 if flags > 0 else -1
            prev_zoom = zoom
            zoom = float(np.clip(zoom + 0.1 * delta, 1.0, 5.0))
            if zoom != prev_zoom:
                cx, cy = x, y
                img_x = int(cx / prev_zoom + x1)
                img_y = int(cy / prev_zoom + y1)
                view_w = int(w / zoom)
                view_h = int(h / zoom)
                new_x1 = img_x - int(cx / zoom)
                new_y1 = img_y - int(cy / zoom)
                offset_x, offset_y = clamp_offset(new_x1, new_y1, view_w, view_h, w, h)

    cv2.setMouseCallback(window_name, mouse_callback)

    help_core = (
        "R = reuse last | S = skip | F = fullscreen | 0 = reset | "
        "+/- = zoom | arrows = pan | z = undo | q = done"
    )
    immediate_key = 0

    while True:
        h_tol = cv2.getTrackbarPos("Hue ±", window_name)
        s_tol = cv2.getTrackbarPos("Sat ±", window_name)
        v_tol = cv2.getTrackbarPos("Val ±", window_name)

        if immediate_key in (ord('r'), ord('R')):
            if last_range is not None:
                lower, upper = last_range
                print(f"↩️ Reusing last HSV range for [{color_name}]: lower={lower.tolist()}, upper={upper.tolist()}")
                cv2.destroyWindow(window_name)
                mask = cv2.inRange(hsv_img, lower, upper)
                count = int(np.count_nonzero(mask))
                total = int(img.shape[0] * img.shape[1])
                ratio = count / total if total > 0 else 0.0
                highlight = img.copy()
                highlight[mask > 0] = config["highlight_color"]
                os.makedirs(output_dir, exist_ok=True)
                cv2.imwrite(os.path.join(output_dir, f"{name_stem}_{color_name.lower()}_highlighted.jpg"), highlight)
                print(f"✅ Processed {name_stem}{ext} [{color_name}] — {round(ratio * 100, 2)}% highlighted")
                row = {
                    "Filename": f"{name_stem}{ext}",
                    f"{color_name}_Pixels": count,
                    "Total_Pixels": total,
                    f"{color_name}_Percentage": round(ratio * 100, 2),
                    "Lower_HSV": lower.tolist(),
                    "Upper_HSV": upper.tolist()
                }
                return row, (lower, upper)
            else:
                print("⚠️ No previous HSV range to reuse.")
                immediate_key = 0

        if immediate_key in (ord('s'), ord('S')):
            print(f"⏭️ Skipped {name_stem}{ext}")
            cv2.destroyWindow(window_name)
            return None, last_range

        if clicked_hsvs:
            hsv_array = np.array(clicked_hsvs)
            lower_rt = np.maximum(np.min(hsv_array, axis=0) - [h_tol, s_tol, v_tol], [0, 0, 0]).astype(int)
            upper_rt = np.minimum(np.max(hsv_array, axis=0) + [h_tol, s_tol, v_tol], [179, 255, 255]).astype(int)
            mask = cv2.inRange(hsv_img, lower_rt, upper_rt)
            highlighted = img.copy()
            highlighted[mask > 0] = config["highlight_color"]
        else:
            highlighted = img.copy()

        h, w = highlighted.shape[:2]
        view_w = int(w / zoom)
        view_h = int(h / zoom)
        offset_x, offset_y = clamp_offset(offset_x, offset_y, view_w, view_h, w, h)
        x1 = offset_x
        y1 = offset_y
        zoomed = highlighted[y1:y1 + view_h, x1:x1 + view_w]
        zoomed = cv2.resize(zoomed, (w, h), interpolation=cv2.INTER_NEAREST)

        max_text_width = w - 20
        overlay = zoomed.copy()
        draw_wrapped_text(
            overlay,
            f"{len(clicked_hsvs)} pt | {help_core}",
            topleft=(10, 10),
            max_width=max_text_width,
            font=FONT,
            scale=FONT_SCALE,
            color=TEXT_COLOR,
            thickness=TEXT_THICK,
            line_gap=LINE_GAP,
            bg_color=BG_COLOR,
            bg_alpha=BG_ALPHA,
            padding=PADDING
        )
        cv2.imshow(window_name, overlay)

        key = cv2.waitKey(1) & 0xFF
        if key in [ord('q'), 27]:
            break
        elif key == ord('z'):
            if clicked_hsvs:
                removed = clicked_hsvs.pop()
                print(f"↩️ Undo: removed HSV {removed}")
            else:
                print("⚠️ No HSV point to undo.")
        elif key in (ord('+'), ord('=')):
            prev_zoom = zoom
            zoom = min(zoom + 0.1, 5.0)
            if zoom != prev_zoom:
                cx, cy = w // 2, h // 2
                img_x = int(cx / prev_zoom + x1)
                img_y = int(cy / prev_zoom + y1)
                view_w = int(w / zoom)
                view_h = int(h / zoom)
                offset_x = img_x - int(cx / zoom)
                offset_y = img_y - int(cy / zoom)
        elif key in (ord('-'), ord('_')):
            prev_zoom = zoom
            zoom = max(zoom - 0.1, 1.0)
            if zoom != prev_zoom:
                cx, cy = w // 2, h // 2
                img_x = int(cx / prev_zoom + x1)
                img_y = int(cy / prev_zoom + y1)
                view_w = int(w / zoom)
                view_h = int(h / zoom)
                offset_x = img_x - int(cx / zoom)
                offset_y = img_y - int(cy / zoom)
        elif key == 82:
            offset_y = max(0, offset_y - int(PAN_STEP / zoom))
        elif key == 84:
            offset_y = min(h - view_h, offset_y + int(PAN_STEP / zoom))
        elif key == 81:
            offset_x = max(0, offset_x - int(PAN_STEP / zoom))
        elif key == 83:
            offset_x = min(w - view_w, offset_x + int(PAN_STEP / zoom))
        elif key in (ord('f'), ord('F')):
            fullscreen = not fullscreen
            if fullscreen:
                cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
            else:
                cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_NORMAL)
                cv2.resizeWindow(window_name, WINDOW_W, WINDOW_H)
                cv2.moveWindow(window_name, WINDOW_X, WINDOW_Y)  # ← re-assert position after exiting FS
        elif key == ord('0'):
            zoom = 1.0
            offset_x, offset_y = 0, 0
        elif key in (ord('r'), ord('R'), ord('s'), ord('S')):
            immediate_key = key
            continue

        immediate_key = 0

    # Final thresholds after interactive session
    h_tol = cv2.getTrackbarPos("Hue ±", window_name)
    s_tol = cv2.getTrackbarPos("Sat ±", window_name)
    v_tol = cv2.getTrackbarPos("Val ±", window_name)
    cv2.destroyWindow(window_name)

    if clicked_hsvs:
        hsv_array = np.array(clicked_hsvs)
        lower = np.maximum(np.min(hsv_array, axis=0) - [h_tol, s_tol, v_tol], [0, 0, 0]).astype(int)
        upper = np.minimum(np.max(hsv_array, axis=0) + [h_tol, s_tol, v_tol], [179, 255, 255]).astype(int)
    else:
        print("⚠️ No HSV selected, using default range.")
        lower = np.array(config["initial_HSV"][:3], dtype=int)
        upper = np.array(config["initial_HSV"][3:], dtype=int)

    mask = cv2.inRange(hsv_img, lower, upper)
    count = int(np.count_nonzero(mask))
    total = int(img.shape[0] * img.shape[1])
    ratio = count / total if total > 0 else 0.0

    highlight = img.copy()
    highlight[mask > 0] = config["highlight_color"]
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(os.path.join(output_dir, f"{name_stem}_{color_name.lower()}_highlighted.jpg"), highlight)

    print(f"✅ Processed {name_stem}{ext} [{color_name}] — {round(ratio * 100, 2)}% highlighted")

    row = {
        "Filename": f"{name_stem}{ext}",
        f"{color_name}_Pixels": count,
        "Total_Pixels": total,
        f"{color_name}_Percentage": round(ratio * 100, 2),
        "Lower_HSV": lower.tolist(),
        "Upper_HSV": upper.tolist()
    }
    return row, (lower, upper)


def main():
    all_imgs = list(iter_images(base_dir, RECURSIVE))
    if not all_imgs:
        print(f"⚠️ No image files found in {base_dir} (recursive={RECURSIVE}).")
        return

    print(f"🔎 Found {len(all_imgs)} image(s).")

    # Per-color state
    last_range: Dict[str, Optional[Tuple[np.ndarray, np.ndarray]]] = {c: None for c in color_configs.keys()}
    results_per_color: Dict[str, List[Dict[str, Any]]] = {c: [] for c in color_configs.keys()}
    checkpoints_per_color: Dict[str, int] = {c: 0 for c in color_configs.keys()}

    # Track which colors are completed per original image
    colors_done_for_image: Dict[str, set] = {}

    total = len(all_imgs)
    num_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    for b in range(num_batches):
        start = b * BATCH_SIZE
        end = min((b + 1) * BATCH_SIZE, total)
        batch_files = all_imgs[start:end]

        # For each color in the desired order, process the batch
        for color_name in COLOR_ORDER:
            cfg = color_configs[color_name]
            out_dir = os.path.join(base_dir, cfg["output_subdir"])
            os.makedirs(out_dir, exist_ok=True)

            # Popup to announce color/batch
            show_popup(
                title=f"Batch {b+1}/{num_batches}",
                message=(
                    f"Now select {color_name.upper()} for images {start+1}–{end} of {total}.\n"
                    "Press any key to start."
                ),
                w=900, h=220, wait_ms=0
            )

            processed_any = False
            for idx, p in enumerate(batch_files, start=start+1):
                p_str = str(p)
                print(f"\n[{idx}/{total}] 🖼 {p.name} [{color_name}]")
                img_bgr = read_image_any(p_str)
                if img_bgr is None:
                    print(f"❌ Could not load {p}")
                    continue

                row, last_tuple = process_image_for_color(
                    img_bgr=img_bgr,
                    img_path=p_str,
                    color_name=color_name,
                    config=cfg,
                    output_dir=out_dir,
                    last_range=last_range[color_name]
                )
                if row is not None:
                    results_per_color[color_name].append(row)
                    processed_any = True
                    # Mark color done for this image
                    colors_done_for_image.setdefault(p_str, set()).add(color_name)

                    # If the image has been processed for ALL colors, post-process original
                    if len(colors_done_for_image[p_str]) == len(color_configs):
                        postprocess_file(Path(p_str))
                        del colors_done_for_image[p_str]

                if last_tuple is not None:
                    last_range[color_name] = last_tuple

            # Save checkpoint for this color after the batch
            checkpoints_per_color[color_name] += 1
            if processed_any:
                save_checkpoint(
                    results=results_per_color[color_name],
                    output_dir=out_dir,
                    color_name=color_name,
                    checkpoint_index=checkpoints_per_color[color_name]
                )

            # Popup to confirm save
            show_popup(
                title=f"{color_name} Saved",
                message=f"Checkpoint saved for {color_name} — Batch {b+1}.\nPress any key to continue.",
                w=900, h=200, wait_ms=0
            )

    # Final save per color
    for color_name, cfg in color_configs.items():
        out_dir = os.path.join(base_dir, cfg["output_subdir"])
        finalize_save(results_per_color[color_name], out_dir, color_name)

    show_popup(
        title="All done!",
        message="All batches processed and final summaries saved.\nPress any key to exit.",
        w=900, h=200, wait_ms=0
    )
    print("\n🏁 All done!")


if __name__ == "__main__":
    main()


🔎 Found 1 image(s).

[1/1] 🖼 cropped_039.jpg [Green]
⚠️ No HSV selected, using default range.
✅ Processed cropped_039.jpg [Green] — 0.0% highlighted
💾 Checkpoint 1 saved for Green: ./output_cropped/Green_Highlighted_PerImage-auto\green_pixel_ratios_per_image_checkpoint_1.xlsx
💾 Checkpoint 1 saved for Green: ./output_cropped/Green_Highlighted_PerImage-auto\green_pixel_ratios_per_image_checkpoint_1.csv

[1/1] 🖼 cropped_039.jpg [Blue]
⚠️ No HSV selected, using default range.
✅ Processed cropped_039.jpg [Blue] — 0.0% highlighted
📁 Moved original to: output_cropped\Processed_Originals\cropped_039.jpg
💾 Checkpoint 1 saved for Blue: ./output_cropped/Blue_Highlighted_PerImage-auto\blue_pixel_ratios_per_image_checkpoint_1.xlsx
💾 Checkpoint 1 saved for Blue: ./output_cropped/Blue_Highlighted_PerImage-auto\blue_pixel_ratios_per_image_checkpoint_1.csv
📊 Summary saved for Green to: ./output_cropped/Green_Highlighted_PerImage-auto\green_pixel_ratios_per_image.xlsx
📄 CSV saved for Green to: ./output_